# Lab 11: 3D Genome Structure - Introduction (pt.2)

Data source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM935611

- **Exercise 1**: TAD calling (contd. from prev. week)
- **Exercise 2**: ChIP-seq downstream analysis

In [ ]:
#!uv add pyranges
#!uv add pyBigWig  # standard, but MIGHT NOT WORK ON WINDOWS:( -> use bedGraph file instead

# import pyBigWig
import pyranges as pr
import cooler
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def plot_hic_matrix(mat, title="Hi-C Contact Map", cmap='Reds', max_color=None, show=True):
    """Plot heatmap of Hi-C matrix"""
    plt.figure(figsize=(8, 8))
    if isinstance(max_color, float):
        max_color = np.percentile(mat, max_color)    
    plt.imshow(mat, origin='upper', cmap=cmap, vmax=max_color)
    plt.colorbar(label="Contact strength")
    plt.title(title)
    plt.xlabel("Bin")
    plt.ylabel("Bin")
    plt.tight_layout()
    if show:        
        plt.show()

def fetch_region_cooler(mcool_path, chrom, start, end, resolution):
    if chrom.startswith('chr'):
        chrom = chrom[3:]  # remove 'chr' prefix for cooler
    c = cooler.Cooler(f"{mcool_path}::resolutions/{resolution}")
    return c.matrix(balance=False).fetch(f"{chrom}:{start}-{end}")

In [ ]:
_mcool_file = '../notebooks-solutions/lab_outputs/rao/GSE63525_GM12878_insitu_primary+replicate_combined.mcool'

sample_region_small = ('chr2', 134_000_000, 138_000_000)
sample_region_small_hic = fetch_region_cooler(_mcool_file, *sample_region_small, 25_000)
plot_hic_matrix(sample_region_small_hic, title="Hi-C Contact Map", max_color=99.0)

In [ ]:
sample_region_big = ('chr2', 120_000_000, 160_000_000)
sample_compartment_resolution = 100_000
sample_region_big_hic_mat = fetch_region_cooler(_mcool_file, *sample_region_big, sample_compartment_resolution)

plot_hic_matrix(sample_region_big_hic_mat, title="Hi-C Contact Map for Larger Region", max_color=99.0)

---
## Exercise 1: TAD calling (contd.)

**Goals:**
1. Detect TAD boundaries using insulation score

In [ ]:
def compute_insulation_score(mat, window=5):
    """Compute insulation score for TAD boundary detection."""
    n = mat.shape[0]
    insulation = np.zeros(n)
    for i in range(n):
        start = max(0, i-window)
        end = min(n, i+window+1)
        submat = mat[start:end, start:end]
        insulation[i] = submat.sum()
    return insulation


def detect_tad_boundaries(insulation, smooth=3):
    """Detect TAD boundaries as local minima in insulation score."""
    n = len(insulation)
    print(n)
    if smooth:
        ins_cs = np.cumsum(insulation)
        ins_cs[smooth:] = ins_cs[smooth:] - ins_cs[:-smooth]
        insulation[smooth//2:-smooth//2+1] = ins_cs[smooth - 1:] / smooth
    boundaries = [
        i for i in range(1, n-1)
        if insulation[i] < insulation[i-1] and insulation[i] < insulation[i+1]
    ]
    return boundaries


def get_tad_regions(boundaries, n_bins):
    """Convert TAD boundaries to a list of TAD regions (start, end)."""
    regions = []
    start = 0
    for b in boundaries:
        regions.append((start, b))
        start = b
    regions.append((start, n_bins))
    return regions

In [ ]:
insulation_score = compute_insulation_score(sample_region_small_hic)
tad_boundaries = detect_tad_boundaries(insulation_score)
print(f"Found {len(tad_boundaries)} TAD boundaries")
tad_regions = get_tad_regions(tad_boundaries, insulation_score.shape[0])
print(f"Extracted {len(tad_boundaries)} TADs")

plt.figure(figsize=(12, 3))
plt.plot(insulation_score);
plt.vlines(tad_boundaries, ymin=insulation_score.min(), ymax=insulation_score.max(), color='orange', linewidth=1);

In [ ]:
from matplotlib.patches import Rectangle

plot_hic_matrix(sample_region_small_hic, title="Hi-C Contact Map", max_color=99.0, show=False)
ax = plt.gca()
for i, (tad_start, tad_end) in enumerate(tad_regions):
    tad_size = tad_end - tad_start
    ax.add_patch(
        Rectangle((tad_start, tad_start), tad_size, tad_size,
                  edgecolor='blue', linewidth=1, fill=False)
    )
    ax.text(tad_start + tad_size, tad_start, str(i))

**TODO**:

Play with windows size and smoothing to get a different result. What else can be done with the raw signal?

---
## Exercise 2: ChIP-seq downstream analysis

**Goals:**
1. Load and visuzlize ChIP-seq signal and peaks.
2. Intersect the peaks with the motifs - use `pyranges`

_Note: The "peak calling" step is outside the scope of this class_

In [ ]:
chipseq_signal = pr.read_bed("../notebooks-solutions/lab_outputs/GSM935611_chr2.bedGraph.gz")  # works fine despite name
chipseq_signal.columns = ['Chromosome', "Start", "End", "Score"]  # for some reason the Score columns is present as "Name"
chipseq_signal

In [ ]:
chipseq_peaks = pr.read_bed("../notebooks-solutions/lab_outputs/GSM935611_hg19_wgEncodeSydhTfbsGm12878Ctcfsc15914c20StdPk.narrowPeak.gz")
chipseq_peaks

In [ ]:
def plot_region(bw_region, peaks_region, chromosome, start, end):
    bw_region = chipseq_signal[chipseq_signal.Chromosome == chromosome]
    bw_region = bw_region[(bw_region.Start < end) & (bw_region.End > start)]

    chipseq_peaks_region = chipseq_peaks[chipseq_peaks.Chromosome == chromosome]
    chipseq_peaks_region = chipseq_peaks_region[(chipseq_peaks_region.Start < end) & (chipseq_peaks_region.End > start)]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, sharex=True, figsize=(12, 4),
        gridspec_kw={"height_ratios": [3, 1]}
    )

    # ---- Signal track ----
    for row in bw_region.df.itertuples():
        ax1.plot([row.Start, row.End], [row.Score, row.Score])

    ax1.set_ylabel("Signal")

    # ---- Peaks track ----
    for row in peaks_region.df.itertuples():
        ax2.axvspan(row.Start, row.End, alpha=0.5)

    ax2.set_xlim(start, end)
    ax2.set_ylabel("Peaks")
    ax2.set_xlabel(f"chr1:{start}-{end}")

    plt.tight_layout()
    plt.show()

plot_region(
    chipseq_signal,
    chipseq_peaks,
    *sample_region_small
)

In [ ]:
d = 20_000  # example - try different values
resolution = 25_000
_df = pd.DataFrame.from_dict({
    'Chromosome': sample_region_small[0],
    'Start': tad_boundaries,
    'End': tad_boundaries,
    'TadIdx': range(1, len(tad_boundaries) + 1)
})
_df['Start'] = _df['Start'] * resolution - d
_df['End'] = _df['End'] * resolution + d
tad_boundaries_pr = pr.PyRanges(_df)
tad_boundaries_pr

In [ ]:
tad_boundaries_pr.join(chipseq_peaks)

In [ ]:
plot_hic_matrix(sample_region_small_hic, title="Hi-C Contact Map", max_color=99.0, show=False)
ax = plt.gca()
for i, (tad_start, tad_end) in enumerate(tad_regions):
    tad_size = tad_end - tad_start
    ax.add_patch(
        Rectangle((tad_start, tad_start), tad_size, tad_size,
                  edgecolor='blue', linewidth=1, fill=False)
    )
    ax.text(tad_start + tad_size, tad_start, str(i))

# TODO: plot the results on the heatmap